# 스트레스 점수 예측 — Public 최고 모델 검증본

**Weighted Quantile ExtraTrees**

- Public MAE: **0.128277**
- ExtraTrees 1,200개, `random_state=42`
- 중요 피처 8개를 각각 두 번 추가
- 트리 예측의 51% 분위수 사용
- 5-Fold 교차검증과 최종 제출 생성을 한 파일에서 재현

In [8]:
from pathlib import Path
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

## 1. 설정 및 데이터

In [ ]:
c
DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project")

TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 1200
QUANTILE = 0.51

WEIGHTED_FEATURES = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio",
    "bone_density",
]

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print("Train:", train.shape)
print("Test:", test.shape)

Train: (3000, 18)
Test: (3000, 17)


## 2. 행 단위 파생변수

In [10]:
def add_features(frame):
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]

    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = (
        result["cholesterol"] / (result["glucose"] + 1e-6)
    )
    result["missing_count"] = result.isna().sum(axis=1)

    for feature in WEIGHTED_FEATURES:
        result[f"{feature}__copy1"] = result[feature]
        result[f"{feature}__copy2"] = result[feature]
    return result


X = add_features(train.drop(columns=[TARGET, ID_COLUMN]))
X_test = add_features(test.drop(columns=[ID_COLUMN]))
y = train[TARGET]

print("Features:", X.shape)

Features: (3000, 38)


## 3. Train 전용 전처리

In [11]:
def make_preprocessor(frame):
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()

    return ColumnTransformer(
        [
            (
                "categorical",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="constant",
                                fill_value="missing",
                            ),
                        ),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore",
                                sparse_output=False,
                            ),
                        ),
                    ]
                ),
                categorical,
            ),
            (
                "numerical",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                ),
                numerical,
            ),
        ]
    )


def predict_quantile(model, matrix):
    tree_predictions = np.column_stack(
        [tree.predict(matrix) for tree in model.estimators_]
    )
    return np.clip(
        np.quantile(tree_predictions, QUANTILE, axis=1),
        0.0,
        1.0,
    )

## 4. 누수 없는 5-Fold 교차검증

In [12]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
oof = np.zeros(len(X))
fold_mae = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X), start=1):
    preprocessor = make_preprocessor(X.iloc[train_idx])
    fold_train = preprocessor.fit_transform(X.iloc[train_idx])
    fold_valid = preprocessor.transform(X.iloc[valid_idx])

    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS,
        min_samples_leaf=1,
        max_features=1,
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    model.fit(fold_train, y.iloc[train_idx])
    prediction = predict_quantile(model, fold_valid)
    oof[valid_idx] = prediction

    score = mean_absolute_error(y.iloc[valid_idx], prediction)
    fold_mae.append(score)
    print(f"Fold {fold} MAE: {score:.6f}")

fold_mae = np.array(fold_mae)
cv_mae = mean_absolute_error(y, oof)

print("Fold MAE:", np.round(fold_mae, 6))
print("Mean CV MAE:", f"{cv_mae:.6f}")
print("MAE Std:", f"{fold_mae.std():.6f}")

Fold 1 MAE: 0.136627
Fold 2 MAE: 0.150506
Fold 3 MAE: 0.149009
Fold 4 MAE: 0.159168
Fold 5 MAE: 0.150243
Fold MAE: [0.136626 0.150506 0.149009 0.159168 0.150242]
Mean CV MAE: 0.149110
MAE Std: 0.007214


## 5. 전체 Train 학습 및 Test 예측

In [13]:
preprocessor = make_preprocessor(X)
train_matrix = preprocessor.fit_transform(X)
test_matrix = preprocessor.transform(X_test)

final_model = ExtraTreesRegressor(
    n_estimators=N_ESTIMATORS,
    min_samples_leaf=1,
    max_features=1,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
final_model.fit(train_matrix, y)
pred_best = predict_quantile(final_model, test_matrix)

print("Prediction shape:", pred_best.shape)
print("Prediction range:", pred_best.min(), pred_best.max())
print("첫 10개 예측값:", pred_best[:10])

Prediction shape: (3000,)
Prediction range: 0.0 1.0
첫 10개 예측값: [0.56 0.97 0.19 0.44 0.53 0.46 0.78 0.47 0.55 0.88]


## 6. 제출 파일 저장

In [14]:
verified_submission = submission.copy()
verified_submission[ID_COLUMN] = test[ID_COLUMN]
verified_submission[TARGET] = pred_best

output_path = OUTPUT_DIR / "submit_best_verified.csv"
verified_submission.to_csv(output_path, index=False, encoding="utf-8")

print("Saved:", output_path)
print("Rows:", len(verified_submission))
verified_submission.head(10)

Saved: /Users/bitsaem/Desktop/stress_project/submit_best_verified.csv
Rows: 3000


,ID,stress_score
0,TEST_0000,0.56
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.44
4,TEST_0004,0.53
5,TEST_0005,0.46
6,TEST_0006,0.78
7,TEST_0007,0.47
8,TEST_0008,0.55
9,TEST_0009,0.88
